In [72]:
import os
import streamlit as st
import pickle
import time
import langchain

# 1. Groq integration comes from langchain_groq
from langchain_groq import ChatGroq

# 1b. Embeddings run locally via HuggingFace (no OpenAI needed)
from langchain_huggingface import HuggingFaceEmbeddings

# 2. Vectorstores have moved to langchain_community
from langchain_community.vectorstores import FAISS

from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_classic.chains.qa_with_sources.loading import load_qa_with_sources_chain

# 4. Text splitters have their own dedicated package
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 5. Document loaders have moved to langchain_community
from langchain_community.document_loaders import UnstructuredURLLoader

In [73]:
import os

os.environ['GROQ_API_KEY'] = "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"

In [74]:
# Initialise LLM with required params
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
    max_tokens=500
)

In [75]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

In [76]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)

In [77]:
len(docs)

19

In [78]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nLoans @ 9.99% in Just 5 Mins\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol AD Lite\n\nMoneycontrol AD Lite\n\nMoneycontrol PRO\n\nMoneycontrol Super PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTren

In [57]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [59]:
from langchain_community.vectorstores import FAISS

# build the FAISS index from your docs + huggingface embeddings
vectorindex = FAISS.from_documents(docs, embeddings)

# save it locally
file_path = "vector_index.pkl"
with open(file_path, "wb") as f:
    pickle.dump(vectorindex, f)
    

In [60]:
if os.path.exists(file_path):
    with open(file_path,"rb") as f:
        vectorIndex=pickle.load(f)

In [65]:
import pickle

file_path = "vector_index.pkl"

with open(file_path, "rb") as f:
    vectorstore = pickle.load(f)

In [80]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.9, max_tokens=500)

prompt = ChatPromptTemplate.from_template("""
Answer the user's question using only the context below.
Context:
{context}
Question:
{input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(
    vectorstore.as_retriever(),
    document_chain
)

query = "What is this article about?"  # replace with your actual question

result = retrieval_chain.invoke({"input": query})
print(result["answer"])

This article appears to be about the current state of the economy, specifically inflation and interest rates, and how they may be affected by upcoming data releases and Federal Reserve decisions.
